# NSE Stock Data Scraper - Colab to Google Sheets

This notebook fetches NSE India stock market data and automatically uploads it to Google Sheets.

## Features
- ✅ Fetch all NSE indices with live data
- ✅ Fetch stocks organized by major indices (NIFTY 50, NIFTY BANK, etc.)
- ✅ Fetch stocks organized by sectors (IT, Auto, Pharma, etc.)
- ✅ Automatic upload to Google Sheets with formatting
- ✅ No local setup required - runs entirely in Google Colab

## Instructions
1. Run each cell in order (Shift+Enter)
2. Authenticate with Google when prompted
3. Choose what data to fetch
4. Data will automatically appear in your Google Sheet!

---

## Step 1: Install Required Libraries

Run this cell first. It will install all necessary Python libraries.

In [ ]:
# Install required libraries (this may take 30-60 seconds)
!pip install -q nse-python gspread google-auth google-auth-oauthlib google-auth-httplib2 pandas tqdm

print("✅ All libraries installed successfully!")
print("✅ Ready to proceed to the next step!")

## Step 2: Import Libraries and Setup

Import all required libraries and define helper classes.

In [ ]:
import pandas as pd
from datetime import datetime
from tqdm.notebook import tqdm
import sys

from nse import NSE
from google.colab import auth
import gspread
from google.auth import default

print("✅ All libraries imported successfully!")

## Step 3: Configuration

**IMPORTANT:** Change `SHEET_NAME` to your desired Google Sheet name!

In [ ]:
# ⚙️ CONFIGURATION - Change this to your Google Sheet name
SHEET_NAME = "NSE Stock Data"  # 👈 CHANGE THIS!

# Major indices to fetch
MAJOR_INDICES = [
    'NIFTY 50', 'NIFTY BANK', 'NIFTY IT', 'NIFTY AUTO',
    'NIFTY PHARMA', 'NIFTY FMCG', 'NIFTY METAL', 'NIFTY REALTY',
    'NIFTY ENERGY', 'NIFTY MIDCAP 50', 'NIFTY SMALLCAP 50',
    'NIFTY 100', 'NIFTY 200', 'NIFTY 500'
]

# Sector indices to fetch
SECTOR_INDICES = [
    'NIFTY BANK', 'NIFTY IT', 'NIFTY AUTO', 'NIFTY PHARMA',
    'NIFTY FMCG', 'NIFTY METAL', 'NIFTY REALTY', 'NIFTY ENERGY',
    'NIFTY FINANCIAL SERVICES', 'NIFTY MEDIA', 'NIFTY PSU BANK',
    'NIFTY PRIVATE BANK', 'NIFTY HEALTHCARE INDEX',
    'NIFTY CONSUMER DURABLES', 'NIFTY OIL & GAS'
]

print(f"📋 Configuration set!")
print(f"📊 Google Sheet Name: {SHEET_NAME}")
print(f"📈 Will fetch data from {len(MAJOR_INDICES)} indices")
print(f"🏭 Will fetch data from {len(SECTOR_INDICES)} sectors")

## Step 4: Define Helper Classes

These classes handle NSE data fetching and Google Sheets uploading.

In [ ]:
class NSEColab:
    """NSE Stock Data Scraper for Google Colab"""

    def __init__(self):
        print("🔄 Initializing NSE Scraper...")
        self.nse = NSE()
        print("✅ NSE Scraper initialized")

    def get_all_indices(self):
        """Fetch all NSE indices"""
        print("📊 Fetching all indices...")
        indices_data = self.nse.get_indices()
        if not indices_data:
            return pd.DataFrame()
        df = pd.DataFrame(indices_data)
        print(f"✅ Fetched {len(df)} indices")
        return df

    def get_index_stocks(self, index_name):
        """Fetch stocks for specific index"""
        try:
            stocks_data = self.nse.get_index_stocks(index_name)
            if not stocks_data:
                return pd.DataFrame()
            df = pd.DataFrame(stocks_data)
            df['Index'] = index_name
            return df
        except Exception as e:
            print(f"⚠️ Error fetching {index_name}: {e}")
            return pd.DataFrame()

    def fetch_all_stocks_by_indices(self, indices=None):
        """Fetch stocks from multiple indices"""
        if indices is None:
            indices = MAJOR_INDICES

        all_stocks = []
        print(f"📈 Fetching stocks from {len(indices)} indices...")

        for index_name in tqdm(indices, desc="Fetching"):
            df = self.get_index_stocks(index_name)
            if not df.empty:
                all_stocks.append(df)

        if not all_stocks:
            return pd.DataFrame()

        combined_df = pd.concat(all_stocks, ignore_index=True)
        print(f"✅ Total stocks: {len(combined_df)}")
        return combined_df

    def fetch_all_stocks_by_sectors(self):
        """Fetch stocks by sectors"""
        all_stocks = []
        print(f"🏭 Fetching stocks from {len(SECTOR_INDICES)} sectors...")

        for sector in tqdm(SECTOR_INDICES, desc="Fetching"):
            df = self.get_index_stocks(sector)
            if not df.empty:
                df['Sector'] = sector
                all_stocks.append(df)

        if not all_stocks:
            return pd.DataFrame()

        combined_df = pd.concat(all_stocks, ignore_index=True)
        print(f"✅ Total stocks: {len(combined_df)}")
        return combined_df


class GoogleSheetsUploader:
    """Upload data to Google Sheets"""

    def __init__(self, sheet_name=SHEET_NAME):
        print("🔐 Authenticating with Google...")
        auth.authenticate_user()
        creds, _ = default()
        self.gc = gspread.authorize(creds)
        self.sheet_name = sheet_name

        try:
            self.spreadsheet = self.gc.open(sheet_name)
            print(f"✅ Connected to: {sheet_name}")
        except gspread.SpreadsheetNotFound:
            print(f"📄 Creating new sheet: {sheet_name}")
            self.spreadsheet = self.gc.create(sheet_name)
            print(f"✅ Created: {sheet_name}")

        print(f"🔗 {self.spreadsheet.url}")

    def upload_dataframe(self, df, worksheet_name, clear=True):
        """Upload DataFrame to worksheet"""
        if df.empty:
            print(f"⚠️ No data for {worksheet_name}")
            return

        try:
            try:
                worksheet = self.spreadsheet.worksheet(worksheet_name)
                if clear:
                    worksheet.clear()
            except gspread.WorksheetNotFound:
                worksheet = self.spreadsheet.add_worksheet(
                    title=worksheet_name,
                    rows=len(df) + 100,
                    cols=len(df.columns) + 5
                )

            # Upload data
            data = [df.columns.tolist()] + df.values.tolist()
            worksheet.update('A1', data)

            # Format header
            worksheet.format('A1:Z1', {
                'backgroundColor': {'red': 0.26, 'green': 0.52, 'blue': 0.96},
                'textFormat': {'bold': True, 'foregroundColor': {'red': 1, 'green': 1, 'blue': 1}},
                'horizontalAlignment': 'CENTER'
            })
            worksheet.freeze(rows=1)

            print(f"✅ Uploaded {len(df)} rows to '{worksheet_name}'")

        except Exception as e:
            print(f"❌ Error: {e}")

    def get_url(self):
        return self.spreadsheet.url


print("✅ Helper classes defined!")

---
# 🚀 CHOOSE WHAT TO FETCH

Run **ONE** of the cells below based on what data you want:

---

## Option 1: Fetch All Indices Only

Fetches list of all NSE indices with live market data.

In [ ]:
# 📊 FETCH ALL INDICES
scraper = NSEColab()
uploader = GoogleSheetsUploader()

indices_df = scraper.get_all_indices()
uploader.upload_dataframe(indices_df, "All Indices")

print("\n✅ Done! Open your sheet:")
print(f"🔗 {uploader.get_url()}")

## Option 2: Fetch Stocks by Index

Fetches all stocks organized by major indices (NIFTY 50, NIFTY BANK, etc.).

In [ ]:
# 📈 FETCH STOCKS BY INDEX
scraper = NSEColab()
uploader = GoogleSheetsUploader()

stocks_df = scraper.fetch_all_stocks_by_indices()
uploader.upload_dataframe(stocks_df, "Stocks by Index")

print("\n✅ Done! Open your sheet:")
print(f"🔗 {uploader.get_url()}")

## Option 3: Fetch Stocks by Sector

Fetches all stocks organized by sectors (Banking, IT, Auto, etc.).

In [ ]:
# 🏭 FETCH STOCKS BY SECTOR
scraper = NSEColab()
uploader = GoogleSheetsUploader()

stocks_df = scraper.fetch_all_stocks_by_sectors()
uploader.upload_dataframe(stocks_df, "Stocks by Sector")

print("\n✅ Done! Open your sheet:")
print(f"🔗 {uploader.get_url()}")

## Option 4: Fetch EVERYTHING 🔥

Fetches all indices + all stocks (by index and sector) and uploads to separate sheets.

In [ ]:
# 🔥 FETCH EVERYTHING!
scraper = NSEColab()
uploader = GoogleSheetsUploader()

print("\n1️⃣ Fetching Indices...")
indices_df = scraper.get_all_indices()
uploader.upload_dataframe(indices_df, "All Indices")

print("\n2️⃣ Fetching Stocks by Index...")
stocks_index_df = scraper.fetch_all_stocks_by_indices()
uploader.upload_dataframe(stocks_index_df, "Stocks by Index")

print("\n3️⃣ Fetching Stocks by Sector...")
stocks_sector_df = scraper.fetch_all_stocks_by_sectors()
uploader.upload_dataframe(stocks_sector_df, "Stocks by Sector")

print("\n" + "="*70)
print("✅ ALL DATA UPLOADED SUCCESSFULLY!")
print("="*70)
print(f"\n🔗 Open your Google Sheet:")
print(f"{uploader.get_url()}")

---

## 📝 Notes

- **First time:** You'll be asked to authenticate with Google
- **Sheet Location:** Your Google Sheet will appear in Google Drive
- **Formatting:** Headers are automatically formatted with blue background
- **Updates:** Run the cells again to refresh the data
- **Sharing:** You can share the Google Sheet with others normally

## 🔗 Links

- GitHub: [aakash-code/NSE-Scrap](https://github.com/aakash-code/NSE-Scrap)
- Documentation: See PYTHON_USAGE.md

---

**Disclaimer:** This tool is for informational purposes only. Always verify data from official sources before making investment decisions.
